# Comparing candidates, with evidence

Now the real job: for every requirement in the JD, essential or preferred,
does this candidate meet it - and why? `evaluate_candidate` answers per
requirement, quoting the resume, and separately calling out transferable
skills (different technology, same underlying capability), missing
information, and internal inconsistencies. `rank_candidates` sorts the
result and flags ties instead of silently breaking them.

## Step 1 - extract the JD and every sample resume

In [ ]:
import os, getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("LITELLM_API_KEY"):
    os.environ["LITELLM_API_KEY"] = getpass.getpass("LiteLLM API key: ")

from recruiting import (
    extract_candidate,
    extract_job_description,
    evaluate_candidate,
    rank_candidates,
)

SAMPLE_DIR = Path("sample_data")

jd = extract_job_description(
    (SAMPLE_DIR / "job_description.txt").read_text(encoding="utf-8"), source_file="job_description.txt"
)

resume_files = sorted(SAMPLE_DIR.glob("resume_*.txt"))
candidates = [
    extract_candidate(path.read_text(encoding="utf-8"), source_file=path.name) for path in resume_files
]
print(f"Loaded the JD and {len(candidates)} candidates:", [c.name for c in candidates])

## Step 2 - evaluate each candidate against the JD

In [2]:
evaluations = [evaluate_candidate(c, jd) for c in candidates]
ranked = rank_candidates(evaluations)

for e in ranked:
    met, total = e.essential_coverage()
    tie = "  (tied with the next candidate)" if e.tied_with_next else ""
    print(f"{e.score:5.1f}  {e.candidate.name:20s}  essential {met}/{total}{tie}")

 95.0  Amina Hassan          essential 5/5
 42.0  Karim El-Sayed        essential 3/5  (tied with the next candidate)
 42.0  Lina Farouk           essential 4/5


## Step 3 - the evidence behind the top score

In [3]:
top = ranked[0]
print(top.candidate.name, "-", top.score, "\n")
for m in top.matches:
    flag = "YES" if m.matched else "NO "
    kind = " (transferable)" if m.transferable else ""
    print(f"[{flag}] [{'essential' if m.essential else 'preferred'}]{kind} {m.requirement}")
    if m.evidence:
        print(f"        evidence: {m.evidence}")
    if m.note:
        print(f"        note: {m.note}")

Amina Hassan - 95.0 

[YES] [essential] 3+ years of professional software engineering experience
        evidence: Backend engineer with 5 years of experience... Backend Engineer, PayFlow Egypt (Jun 2021 to Present) and Software Engineer, Cairo Byte Labs (Jul 2019 to May 2021)
        note: 5 years total experience clearly exceeds 3+ year requirement
[YES] [essential] Strong proficiency in Python
        evidence: Built internal REST APIs in Python (Flask)... Skills: Python, Django, Flask
        note: Python listed as primary skill with demonstrated use in multiple roles
[YES] [essential] Experience designing and consuming REST APIs
        evidence: Designed and shipped REST APIs (Django REST Framework)... Built internal REST APIs in Python (Flask)... REST API design listed in skills
        note: Extensive REST API design experience across two roles
[YES] [essential] Experience with relational databases (PostgreSQL, MySQL)
        evidence: Owned the PostgreSQL schema for the transa

## Step 4 - a transferable-skill match, and a set of red flags

Karim's background is Ruby on Rails, not Python - watch for
`transferable=True` matches on the Python/REST requirements. Lina's resume
has gaps: look at `missing_info` and `inconsistencies`.

In [4]:
karim = next(e for e in ranked if e.candidate.name.split()[0] == "Karim")
print("Karim - transferable matches:")
for m in karim.matches:
    if m.transferable:
        print(" -", m.requirement, "->", m.note)

lina = next(e for e in ranked if e.candidate.name.split()[0] == "Lina")
print("\nLina - missing info:", lina.missing_info)
print("Lina - inconsistencies:", lina.inconsistencies)

Karim - transferable matches:

Lina - missing info: ["No end date provided for 'Freelance Web Developer' role; unclear how long this lasted or if it overlaps with other positions", 'No specific technologies or frameworks mentioned for backend work (e.g., Django, Flask, FastAPI)', "No specific relational databases named (PostgreSQL, MySQL); only generic 'SQL' and 'database-related tasks' mentioned", 'No details on scope or complexity of API work (design vs. consumption, REST vs. other types)', 'No GPA, graduation date, or honors mentioned for degree', 'Vague descriptions of responsibilities; limited quantifiable achievements or impact']
Lina - inconsistencies: ['Employment timeline overlap: Backend Developer (Mar 2021–Nov 2023) and Full-Stack Developer (Aug 2022–Present) overlap from Aug 2022 to Nov 2023. Resume does not clarify if these were concurrent or if dates are incorrect.', "Unclear total professional experience: If roles overlapped, actual experience is ~2.8 years; if sequentia

## What just happened

Every score comes with a paper trail: which requirement, matched or not,
quoted from the resume, and whether it was a literal or a transferable
match. Nothing here picks a winner on the recruiter's behalf - it hands over
a ranked list with the reasoning attached.

Your turn: edit `sample_data/job_description.txt` - move Docker from
preferred to essential - and re-run this notebook. Nothing else changes;
the ranking updates because the requirements did. That is the same move the
live agent makes in notebook 04 when a JD changes mid-demo.